In [ ]:
import sqlite3
import requests
import re

LLM_URL = "http://localhost:1234/v1/chat/completions"
MODEL_NAME = "local-model"  # LM Studio에서 실행 중인 모델명으로 변경


# 1. 샘플 DB 생성
def init_db():
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE sales (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        branch TEXT,
        product_name TEXT,
        sales_amount INTEGER,
        sales_date TEXT
    )
    """)

    cur.executemany("""
    INSERT INTO sales (branch, product_name, sales_amount, sales_date)
    VALUES (?, ?, ?, ?)
    """, [
        ("서울", "사과", 12000, "2026-04-01"),
        ("서울", "바나나", 8000, "2026-04-02"),
        ("부산", "사과", 15000, "2026-04-03"),
        ("서울", "딸기", 22000, "2026-04-04"),
    ])

    conn.commit()
    return conn


# 2. DB 스키마 가져오기
def get_db_schema(conn):
    cur = conn.cursor()

    cur.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    """)

    tables = cur.fetchall()
    schema_text = ""

    for (table_name,) in tables:
        schema_text += f"\nTABLE: {table_name}\n"

        cur.execute(f"PRAGMA table_info({table_name})")
        columns = cur.fetchall()

        for col in columns:
            col_name = col[1]
            col_type = col[2]
            schema_text += f"- {col_name}: {col_type}\n"

    return schema_text


# 3. LLM 호출
def call_llm(prompt):
    response = requests.post(
        LLM_URL,
        json={
            "model": MODEL_NAME,
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "You are a Text-to-SQL assistant. "
                        "Return only SQL. Do not explain. "
                        "Only generate SELECT queries."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "temperature": 0.1
        }
    )

    response.raise_for_status()
    data = response.json()

    return data["choices"][0]["message"]["content"].strip()


# 4. SQL 생성
def generate_sql(user_question, schema):
    prompt = f"""
다음 DB 스키마를 참고해서 사용자 질문에 맞는 SQLite SQL을 작성해줘.

[DB Schema]
{schema}

[User Question]
{user_question}

규칙:
- SELECT 문만 작성해.
- INSERT, UPDATE, DELETE, DROP, ALTER는 절대 사용하지 마.
- 설명 없이 SQL만 반환해.
"""

    sql = call_llm(prompt)

    # 혹시 ```sql 코드블록으로 온 경우 제거
    sql = sql.replace("```sql", "").replace("```", "").strip()

    return sql


# 5. SQL 안전 검증
def validate_sql(sql):
    dangerous_keywords = [
        "insert", "update", "delete", "drop", "alter",
        "truncate", "create", "replace"
    ]

    normalized = sql.lower().strip()

    if not normalized.startswith("select"):
        raise ValueError("SELECT 문만 실행할 수 있습니다.")

    for keyword in dangerous_keywords:
        if re.search(rf"\b{keyword}\b", normalized):
            raise ValueError(f"위험한 SQL 키워드가 포함되어 있습니다: {keyword}")

    return True


# 6. SQL 실행
def execute_sql(conn, sql):
    cur = conn.cursor()
    cur.execute(sql)

    columns = [desc[0] for desc in cur.description]
    rows = cur.fetchall()

    return columns, rows


# 7. 전체 실행
if __name__ == "__main__":
    conn = init_db()

    schema = get_db_schema(conn)
    print("[DB Schema]")
    print(schema)

    user_question = "서울 지점에서 매출이 가장 높은 상품을 보여줘"

    sql = generate_sql(user_question, schema)
    print("\n[Generated SQL]")
    print(sql)

    validate_sql(sql)

    columns, rows = execute_sql(conn, sql)

    print("\n[Result]")
    print(columns)
    for row in rows:
        print(row)